In [ ]:
# --- Célula Extra: Real-time Metrics Dashboard ---
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col, count
import time

# Inicializar SparkSession
spark = SparkSession.builder.appName("RealtimeMetrics").getOrCreate()

# Simular stream de eventos (em produção, viria do Kafka)
schema = "timestamp timestamp, user_id int, action string"

# Criar stream fake
fake_stream = spark.readStream.format("rate").option("rowsPerSecond", 100).load() \
.selectExpr("timestamp", "value % 100 as user_id",
"CASE WHEN value % 3 = 0 THEN 'click' ELSE 'view' END as action")

# Agregação em janela de 30 segundos
metrics = fake_stream.groupBy(
window(col("timestamp"), "30 seconds"),
col("action")
).count()

# Escrever para console (em produção, iria para Grafana/Kibana)
query = metrics.writeStream \
.outputMode("complete") \
.format("console") \
.option("truncate", False) \
.start()

# Deixar rodar por 2 minutos
time.sleep(120)
query.stop()

# Parar a SparkSession
spark.stop()

print("Dashboard em tempo real executado!")


Dashboard em tempo real executado!


### Explicação do Dashboard em Tempo Real

Nesta célula anterior, utilizamos o **Spark Structured Streaming** para simular um ambiente de processamento de dados contínuo. Aqui estão os pontos principais:

1.  **Simulação de Dados (`readStream`)**: Utilizamos o formato `rate` do Spark para gerar dados automaticamente (100 linhas por segundo). Isso simula uma fonte de dados externa, como o Apache Kafka.
2.  **Transformação de Dados**: Criamos colunas fictícias de `user_id` e `action` (cliques ou visualizações) usando expressões SQL para dar contexto aos dados.
3.  **Janelamento (`window`)**: Agrupamos os dados em janelas de tempo de 30 segundos. Isso é essencial em streaming para analisar tendências recentes em vez de todo o histórico.
4.  **Saída no Console (`writeStream`)**: O modo `complete` garante que a tabela de agregação resultante seja impressa inteiramente no console a cada atualização.
5.  **Ciclo de Vida**: O script roda por 120 segundos (`time.sleep`) antes de interromper a query e encerrar a sessão Spark para liberar recursos.